DATABASE DOCUMENTALE E-**COMMERCE**

In [ ]:
from elasticsearch import Elasticsearch

ELASTIC_URL = "INSERIRE_URL_ELASTICSEARCH"
USERNAME = "INSERIRE_USERNAME"
PASSWORD = "INSERIRE_PASSWORD"

es = Elasticsearch(
    ELASTIC_URL,
    basic_auth=(USERNAME, PASSWORD)
)


Anche se Elasticsearch è 'schemaless' (flessibile), definire
esplicitamente il Mapping è fondamentale:
 - 'keyword': viene utilizzato per contenuti strutturati come ID, indirizzi email, nomi host, codici di stato, codici postali o tag.
 - 'text': per indicizzare i valori full-text come il corpo di un'email o descrizione di un prodotto.
 Viene convertita la stringa in un elenco di singoli termini prima di essere indicizzati.
 Tale processo di analisi consente a ElasticSearch di cercare singole parole all'interno di ogni campo di testo completo.
 - diversi tipi di dati.

 Il mapping è il processo di definizione di come un documento e i campi che contiene vengono archiviati e indicizzati -> schema di tabella in un database relazionale.

 Ogni documento è una raccolta di campi, ognuno dei quali ha il proprio tipo di dati.

 La definizione di mappatura contiene un elenco di campi pertinenti al documento.

In [ ]:
#MAPPING/INDICE
id_indice = "catalogo_prodotti"

mapping_ecommerce = {
    "mappings": {
        "properties": {
            "prodotto_id": { "type": "keyword" },
            "modello": { "type": "keyword" },
            "descrizione": { "type": "text" },
            "marca": { "type": "keyword" },
            "categoria": { "type": "keyword" },
            "prezzo": { "type": "float" },
            "quantita_magazzino": { "type": "integer" },
            "recensioni": { "type": "text" },
            "punteggio_soddisfazione": { "type": "float" },
            "data_acquisto": { "type": "date" },
            "data_recensione": { "type": "date" },
            "specifiche_tecniche": { "type": "text" },
            "disponibile": { "type": "boolean" },
            "valutazione_media": { "type": "float" },
            "tags": { "type": "keyword" }
        }
    }
}

Questo indice conterrà un insieme di prodotti di elettronica venduti da un'e-commerce generico e le loro caratteristiche.

Tra le caratteristiche abbiamo:

- prodotto_id, modello, marca, categoria e
tags: si tratta di **keyword** in quanto a differenza dei campi TEXT non vengono analizzate al momento dell'indicizzazione.
**Tags** avrà un array di stringhe.

Questo significa che mentre i campi TEXT sono suddivisi nei loro termini individuali per consentire una corrispondenza parziale mentre i campi valori chiave sono indicizzati cosi come sono.

Ciò vuol dire che è la scelta ottimale per tali campi, in quanto ElasticSearch memorizza il valore esattamente come fornito.

L'utilizzo del tipo di parola chiave ci consente di eseguire aggregazioni sul campo, ordinamento e ottenere informazioni accurate sui documenti.

Essi permettono di identificare in modo univoco ogni prodotto.

- descrizione, recensioni e specifiche tecniche: si tratta di **text**.
Ottimo per la ricerca full-text.

Le altre caratteristiche hanno tipi di dato differenti (float, integer, boolean e date).




In [ ]:
#CREAZIONE DELL'INDICE
if not es.indices.exists(index=id_indice):
    es.indices.create(index=id_indice, body=mapping_ecommerce)
    print(f"indice '{id_indice}'creato")
else:
    print(f"indice '{id_indice}'esiste già")

In [ ]:
#INSERIMENTO DATI CON POST_BULK

dati_bulk = [
    { "index": { "_index": id_indice, "_id": "1" } },
    {
        "prodotto_id": "P001",
        "modello": "Quantum X",
        "descrizione": "Display AMOLED con batteria a lunghissima durata da 5000mAh, ideale per il gaming mobile.",
        "marca": "TechCorp",
        "categoria": "Elettronica",
        "prezzo": 699.99,
        "quantita_magazzino": 50,
        "recensioni": "Telefono superbo, la batteria dura veramente due giorni interi di uso intenso!",
        "punteggio_soddisfazione": 4.8,
        "data_acquisto": "2026-06-10",
        "data_recensione": "2026-06-15",
        "specifiche_tecniche": "RAM 8GB, Memoria Interna 256GB, Processore Octa-Core 2.8GHz",
        "disponibile": True,
        "valutazione_media": 4.7,
        "tags": ["smartphone", "5g", "gaming", "novita"]
    },

    { "index": { "_index": id_indice, "_id": "2" } },
    {
        "prodotto_id": "P002",
        "modello": "SoundPro Wireless",
        "descrizione": "Cuffie sovraurali con cancellazione attiva del rumore (ANC) di livello professionale.",
        "marca": "AudioWave",
        "categoria": "Elettronica",
        "prezzo": 149.50,
        "quantita_magazzino": 120,
        "recensioni": "Isolamento acustico incredibile, comode anche dopo molte ore di utilizzo.",
        "punteggio_soddisfazione": 4.5,
        "data_acquisto": "2026-07-01",
        "data_recensione": "2026-07-03",
        "specifiche_tecniche": "Driver da 40mm, Bluetooth 5.2, Autonomia 40 ore con ANC attivo",
        "disponibile": True,
        "valutazione_media": 4.5,
        "tags": ["audio", "cuffie", "wireless", "noise-cancelling"]
    },

    { "index": { "_index": id_indice, "_id": "3" } },
    {
        "prodotto_id": "P003",
        "modello": "Trekker 3000",
        "descrizione": "Giacca tecnica isolante per alta montagna resistente a vento forte e pioggia battente.",
        "marca": "OutGear",
        "categoria": "Abbigliamento",
        "prezzo": 89.99,
        "quantita_magazzino": 0,
        "recensioni": "Molto calda e impermeabile. Forse un po' rigida nei movimenti, ma fa il suo dovere.",
        "punteggio_soddisfazione": 4.0,
        "data_acquisto": "2026-05-20",
        "data_recensione": "2026-05-25",
        "specifiche_tecniche": "Membrana impermeabile 10.000mm, Cuciture termonastrate, Tasca interna di sicurezza",
        "disponibile": False,
        "valutazione_media": 4.1,
        "tags": ["outdoor", "giacca", "trekking", "impermeabile"]
    }
]


risposta = es.bulk(body=dati_bulk)

if not risposta.get('errors'):
    print("Nuovo inserimento Bulk completato con successo senza errori!")
else:
    print("Attenzione, si sono verificati degli errori durante l'inserimento bulk.")

**QUERY**

In [ ]:
#RICERCA FULL-TEXT
query_1 = {
    "query": {
        "match": {
            "descrizione": "batteria"
        }
    }
}

risultato_1 = es.search(index=id_indice, body=query_1)

#risultati trovati in totale
print("Risultati trovati:", risultato_1["hits"]["total"]["value"])

#id e contenuto completo per ciascun documento trovato
for hit in risultato_1["hits"]["hits"]:
    print(hit["_id"], hit["_source"])

E' una ricerca testuale con cui possiamo trovare prodotti basati su parole chiave specifiche nella loro descrizione ed in questo caso la parola chiave è **batteria**.

Il risultato è un dizionario Python che contiene i risultati restituiti dalla query e il numero di documenti trovati.
Il ciclo for itera attraverso i documenti trovati e restituisce ciascuno con:

-  il loro ID univoco;
- un dizionario Python con tutti i campi del prodotto per ogni documento che corrisponde alla query.


In [ ]:
#FILTRO
query_2 = {
    "query": {
        "bool": {
            "must": { "match": { "categoria": "Elettronica" } },
            "filter": { "range": { "prezzo": { "lt": 200.00 } } }
        }
    }
}

risultato_2 = es.search(index=id_indice, body=query_2)

#risultati trovati in totale
print("Risultati trovati:", risultato_2["hits"]["total"]["value"])

#id e contenuto completo per ogni documento trovato
for hit in risultato_2["hits"]["hits"]:
    print(hit["_id"], hit["_source"])

E' una query booleana:

- must: richiede che il campo categoria dei documenti debba contenere esattemente la parola **"Elettronica"**;
- filter: filtro sul campo prezzo, per cui nei risultati della query verranno inclusi solo i prodotti il cui prezzo è minore di **200.00.**

Otteniamo il risultato della query memorizzato nel dizionario Python e il numero totale dei risultati trovati.
Questo ciclo itera su ogni documento trovato hit e per ciascuno restituirà:
-  il loro id univoco,
- il contenuto completo del documento originale come dizionario Python.

In [ ]:
#chiusura
es.close()